# 07 — Ransomware Leak-Site Detection

Ransomware groups run public "name and shame" sites: they post the victim's name, proof-of-breach snippets, and a countdown to full data release. For a SOC, **an unexpected entry on a leak site is one of the highest-severity signals possible** — it means the breach has already happened, and the conversation is now public.

This notebook does two things:

1. **Binary classifier** — fine-tune DarkBERT on S2W's `leaksite_detection_dataset.tsv` (802 labeled posts, 13% positive) to answer: *is this post announcing a ransomware victim?*
2. **Victim extraction** — for each post the classifier flags as positive, run the nb 05 DarkBERT entity extractor to pull out `ORGANIZATION` spans — the named victim(s).

The composition *classifier → extractor* mirrors how real tooling (DarkFeed, Ransomlook, etc.) works: filter the firehose, then structure what's left.

## Prerequisites

- Hugging Face access to `s2w-ai/DarkBERT` (the benchmark TSV lives in the same repo as the model weights).
- Nb 05 complete — we use its saved entity model for victim extraction.

## 1 — Load the benchmark

In [ ]:
import json, re
from pathlib import Path
import numpy as np, pandas as pd
import torch
from huggingface_hub import hf_hub_download

TSV_PATH = hf_hub_download(repo_id='s2w-ai/DarkBERT',
                            filename='benchmark-dataset/leaksite_detection_dataset.tsv',
                            local_dir='data/darkbert_bench')
df = pd.read_csv(TSV_PATH, sep='\t')
df['label'] = df['label'].astype(int)
df['text_full'] = (df['title'].fillna('') + ' . ' + df['text'].fillna('')).str.strip()

print(f'Rows: {len(df)}')
print(f'Positive (leak announcements): {(df["label"]==1).sum()}  ({(df["label"]==1).mean()*100:.1f}%)')
print(f'Negative: {(df["label"]==0).sum()}')
lens = df['text_full'].str.split().str.len()
print(f'Word count: median={int(lens.median())}, p95={int(lens.quantile(0.95))}, max={lens.max()}')

In [ ]:
# --- Read a positive and a negative to calibrate expectations ---
print('===== POSITIVE example (ransomware leak post) =====')
print(df[df['label']==1].iloc[0]['text_full'][:1000])
print('\n===== NEGATIVE example =====')
print(df[df['label']==0].iloc[0]['text_full'][:1000])

## 2 — Stratified split + tokenize

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from transformers import AutoTokenizer

train_df, temp_df = train_test_split(df, test_size=0.25, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.4, stratify=temp_df['label'], random_state=42)
print(f'train={len(train_df)}  val={len(val_df)}  test={len(test_df)}')
print(f'positive fraction — train: {train_df["label"].mean():.2f}, val: {val_df["label"].mean():.2f}, test: {test_df["label"].mean():.2f}')

MODEL_CKPT = 's2w-ai/DarkBERT'
MAX_LEN = 512
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT, use_fast=True)

feats = Features({'text_full': Value('string'), 'label': ClassLabel(names=['not_leak', 'leak'])})
def to_ds(d):
    return Dataset.from_pandas(d[['text_full', 'label']], features=feats, preserve_index=False)
ds = DatasetDict({'train': to_ds(train_df), 'validation': to_ds(val_df), 'test': to_ds(test_df)})

def tok(batch):
    return tokenizer(batch['text_full'], truncation=True, max_length=MAX_LEN)
tokds = ds.map(tok, batched=True, remove_columns=['text_full'])

## 3 — Fine-tune DarkBERT, with class-weighted loss

13% positive class — using a weighted cross-entropy gets us faster and more reliable recall on the minority class than relying on training-time oversampling.

In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, num_labels=2,
    id2label={0: 'not_leak', 1: 'leak'}, label2id={'not_leak': 0, 'leak': 1},
)

pos_weight = (train_df['label'] == 0).sum() / max((train_df['label'] == 1).sum(), 1)
class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float32)
print(f'Positive class weight: {pos_weight:.2f}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss = torch.nn.functional.cross_entropy(
            logits, labels, weight=class_weights.to(logits.device))
        return (loss, outputs) if return_outputs else loss

def metrics_fn(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = float('nan')
    return {'accuracy': accuracy_score(labels, preds),
            'precision': p, 'recall': r, 'f1': f1, 'roc_auc': auc}

args = TrainingArguments(
    output_dir='models/darkbert_leaksite',
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=tokds['train'], eval_dataset=tokds['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrics_fn,
)
trainer.train()

## 4 — Threshold tuning on validation, evaluate on test

Default 0.5 threshold is rarely optimal on imbalanced data. Sweep thresholds on validation to maximize F1, then apply to test.

In [ ]:
# --- Val probabilities -> pick threshold ---
val_out = trainer.predict(tokds['validation'])
val_probs = torch.softmax(torch.tensor(val_out.predictions), dim=-1).numpy()[:, 1]

best_t, best_f1 = 0.5, 0.0
for t in np.linspace(0.1, 0.9, 33):
    yhat = (val_probs >= t).astype(int)
    f1 = f1_score(val_out.label_ids, yhat, zero_division=0)
    if f1 > best_f1:
        best_t, best_f1 = float(t), float(f1)
print(f'Best threshold on val: {best_t:.2f}  (F1={best_f1:.3f})')

# --- Apply to test ---
test_out = trainer.predict(tokds['test'])
test_probs = torch.softmax(torch.tensor(test_out.predictions), dim=-1).numpy()[:, 1]
test_y = test_out.label_ids
test_yhat = (test_probs >= best_t).astype(int)

print(f'\nTest @ t={best_t:.2f}:')
print(classification_report(test_y, test_yhat, target_names=['not_leak', 'leak'], digits=4))
cm = confusion_matrix(test_y, test_yhat)
print(f'Confusion matrix (rows=true, cols=pred):')
print(f'              not_leak  leak')
print(f'  not_leak      {cm[0,0]:>5d}  {cm[0,1]:>5d}')
print(f'  leak          {cm[1,0]:>5d}  {cm[1,1]:>5d}')
try:
    print(f'\nROC AUC: {roc_auc_score(test_y, test_probs):.4f}')
except Exception as e:
    print(f'ROC AUC: {e}')

## 5 — Extract named victims from positive posts

Loading the nb 05 entity extractor. On each post the classifier flagged positive, we run the extractor and keep `ORGANIZATION` spans — these are candidate victim names.

In [ ]:
from transformers import AutoModelForTokenClassification

ENT_PATH = Path('models/darkbert_entity_final')
if ENT_PATH.exists():
    with open(ENT_PATH / 'tag_vocab.json') as f:
        tag_vocab = json.load(f)
    id2tag = {i: t for i, t in enumerate(tag_vocab['tags'])}
    ent_tok = AutoTokenizer.from_pretrained(str(ENT_PATH), use_fast=True)
    ent_model = AutoModelForTokenClassification.from_pretrained(str(ENT_PATH)).to('cuda').eval()
    entity_available = True
    print('Loaded nb 05 entity extractor.')
else:
    entity_available = False
    print('Nb 05 entity model not found — victim extraction will be skipped.')

def extract_orgs(text, max_len=512, stride=64):
    if not entity_available:
        return []
    enc = ent_tok(text, truncation=True, max_length=max_len,
                  return_offsets_mapping=True, return_overflowing_tokens=True,
                  stride=stride, return_tensors='pt', padding=True).to('cuda')
    offsets = enc.pop('offset_mapping').cpu().numpy()
    enc.pop('overflow_to_sample_mapping', None)
    with torch.no_grad():
        logits = ent_model(**enc).logits.cpu().numpy()
    tags = logits.argmax(-1)
    out = set()
    for chunk_tags, chunk_offs in zip(tags, offsets):
        cur = None
        for (s, e), tid in zip(chunk_offs, chunk_tags):
            if s == e == 0: continue
            tag = id2tag[int(tid)]
            if tag == 'B-ORGANIZATION':
                if cur: out.add(text[cur[0]:cur[1]].strip())
                cur = [int(s), int(e)]
            elif tag == 'I-ORGANIZATION' and cur:
                cur[1] = int(e)
            else:
                if cur: out.add(text[cur[0]:cur[1]].strip())
                cur = None
        if cur: out.add(text[cur[0]:cur[1]].strip())
    # drop very short / garbage spans
    return sorted(x for x in out if 3 <= len(x) <= 80)

# --- Run extractor on test-set positives ---
test_df_reset = test_df.reset_index(drop=True)
pos_mask = (test_yhat == 1)
flagged = test_df_reset[pos_mask].copy()
flagged['true_label'] = test_y[pos_mask]
flagged['score'] = test_probs[pos_mask]
flagged['candidate_victims'] = flagged['text_full'].apply(extract_orgs)
flagged = flagged.sort_values('score', ascending=False)

print(f'Flagged positive in test: {len(flagged)} posts (of which {flagged["true_label"].sum()} true positives)')
print(f'\nTop 10 flagged posts + candidate victims:\n')
for _, r in flagged.head(10).iterrows():
    tag = 'TP' if r['true_label'] == 1 else 'FP'
    title = (r['title'] or '')[:90].replace('\n', ' ')
    print(f'[{tag}] score={r["score"]:.3f}')
    print(f'      title: {title}')
    vics = r['candidate_victims'][:5]
    print(f'      victims: {vics if vics else "(none extracted)"}')
    print()

## 6 — Save the leak-site classifier

In [ ]:
FINAL = Path('models/darkbert_leaksite_final')
FINAL.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL))
tokenizer.save_pretrained(str(FINAL))
with open(FINAL / 'threshold.json', 'w') as f:
    json.dump({'threshold': best_t}, f)
print(f'Saved leak-site classifier (threshold={best_t:.2f}) to {FINAL}')

## What you've done

- Fine-tuned **DarkBERT as a binary classifier** on the bundled leak-site benchmark — the class imbalance (13% positive) handled via class-weighted loss.
- Tuned the decision threshold on validation, reported precision/recall/F1/ROC-AUC on held-out test.
- Chained the classifier with the nb 05 entity extractor so each flagged post produces **candidate victim names** — the structured output a SOC actually needs.

## Honest caveats

- **The benchmark is small** (802 rows). Variance across seeds will be noticeable — results from a single run are directional, not authoritative.
- **Victim extraction is only as good as nb 05's ORG silver labels.** Some victim names (obscure companies, non-English orgs) will be missed.
- **No deduplication of victim names across posts.** A real ransomware tracker normalizes (`"Acme Corp"` ≡ `"Acme Corporation"`) and clusters.

## What's next

- **Nb 08 — Credential dump parser.** The other high-severity dark-web signal: combo lists and stealer logs. Synthetic data + HIBP metadata for real-world context.